1. More on MCP Servers
2. Knowledge Graph memory: It's a persistent memory store of entities, observations about them, and relationships between them. Helps maintain a persistent, structured memory across different chat sessions
3. Creating a web service MCP with [Brave search](https://brave.com/search/api/)
4. the Polygon.io MCP Server

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

### The first type of MCP Server: runs locally, everything local

Knowledge Graph memory:


It's a persistent memory store of entities, observations about them, and relationships between them.

https://github.com/modelcontextprotocol/servers/tree/main/src/memor

In [ ]:
params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/nober.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [26]:
instructions = """
You are an AI agent with persistent long-term memory powered by entity tools.

Rules for memory:
- Whenever the user tells you any personal information (name, job, preferences, what they're doing, teaching a course, etc.), immediately use the available memory tools to STORE it.
- Prefer tools like `store_entity`, `create_entity`, `add_fact`, `store_knowledge` or similar entity-related tools.
- After storing, you can confirm briefly if needed.
- When the user asks "what do you know about me?" or "recall information about X", first use the recall/search tools (`recall_entities`, `search_memory`, `get_entity`, etc.) to fetch the information, then answer based on what you retrieved.
- Always try to use memory tools before saying you don't know something about the user.
- Do not hallucinate or make up information — only use what the memory tools return.

You have access to these memory tools via MCP.
"
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4.1-mini"""

request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4.1-mini"

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
# This checks whether the memory is persistent, the agent will use search_node tool to check if the entity is stored and add this to context

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

### The 2nd type of MCP server - runs locally, calls a web service

### Creating a web service MCP with [Brace search](https://brave.com/search/api/)

In [ ]:
env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}
params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": env}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
web_search_params = {
    "command": "npx",
    "args": ["-y", "one-search-mcp"],   # or the exact package name from GitHub
    "env": {}
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=120) as server:
    tools = await server.list_tools()




tools


In [ ]:
for tool in tools:
    print(f"{tool.name}: {tool.description}")

In [42]:
instructions = """You are able to search the web for information and briefly summarize the takeaways in 2 lines with links to where the information was found.
Always use english when calling out one_search, return only text results

"""
request = f"Who is the current ps for security in kenya. For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4o-mini"

In [43]:
web_search_params = {
    "command": "npx",
    "args": ["-y", "one-search-mcp"],   # or the exact package name from GitHub
    "env": {}
}

async with MCPServerStdio(params=web_search_params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

As of March 2026, the current Principal Secretary for Security in Kenya is **Julius Korir**. He has been actively involved in overseeing the country’s security affairs. For more details, you can check [source](https://www.bing.com/ck/a).


Anthropic lists some remote MCP servers, but these are for paid applications with business users:

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

CloudFlare has tooling for you to create and deploy your own remote MCP servers, but this does not seem to be a common practice:

https://developers.cloudflare.com/agents/guides/remote-mcp-server/

## The Polygon.io MCP Server

### And now the third type: running remotely

In [ ]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY is not set")

In [ ]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AMZN")[0]

In [ ]:
from market import get_share_price
get_share_price("AMZN")

In [ ]:
# no rate limiting concerns!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

In [ ]:
stock_market_params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()


In [ ]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

### Polygon paid plans have a rate limit of 1000 requests per day.

In [ ]:
params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

In [ ]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple? Use your get_snapshot_ticker tool to get the latest price."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Setting up your .env file for Paid Plans

If you do decide to have a paid plan, please add this to your .env file to indicate:

`POLYGON_PLAN=paid`

And if you decide to go all the way for the realtime API, then please do:

`POLYGON_PLAN=realtime`

In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("You've chosen to subscribe to the paid Polygon plan, so the code will look at prices on a 15 min delay")
elif is_realtime_polygon:
    print("Wowzer - you've chosen to subscribe to the realtime Polygon plan, so the code will look at realtime prices")
else:
    print("According to your .env file, you've chosen to subscribe to the free Polygon plan, so the code will look at EOD prices")

  <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Explore MCP server marketplaces and integrate your own, using all 3 approaches.
            </span>